In [29]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tifffile import imsave
import os
from scipy import signal
from skimage.restoration import unwrap_phase
from skimage import exposure
import io
import imageio
from scipy import stats
import numpy as geek
import cv2

In [30]:

def compress_matrix(matrix, compression_ratio = 5):
    """Reduce a matrix by a factor, called compression_ratio, in each dimension. 
    Each new entry is the average value in a block of size compression_ratio^2"""
    m,n = matrix.shape
    R = int(np.floor(compression_ratio)) # compression ratio must be a positive integer
    
    square = np.ones((R,R))/(R**2) #used to compute the mean value in each block of size RxR
    convoluted_matrix = signal.convolve2d(matrix,square, boundary='symm', mode='same')
    
    compressed_m = int(np.floor(m/R))
    compressed_n = int(np.floor(n/R))

    compressed_matrix = np.zeros((compressed_m,compressed_n))
    for new_index_i in range(compressed_m ):
        original_index_i = new_index_i *compression_ratio
        for new_index_j in range(compressed_n):
            original_index_j = new_index_j *compression_ratio
            compressed_matrix[new_index_i, new_index_j] = convoluted_matrix[original_index_i,original_index_j]
    
    return compressed_matrix
        
def compress_tiff(tifffilepath, compression_ratio = 5):
    """ Compress each frame in a tiff file, located at tiffpath, by a factor compression_ratio
    vertically and horizontally. The resulting tiff file is saved with extension _compressed_ratio"""
    
    filename, file_extension = os.path.splitext(tifffilepath)
    
    # load data
    tif = tifffile.TiffFile(tifffilepath)
    
    R = int(np.floor(compression_ratio)) # compression ratio must be a positive integer
    N = len(tif.pages) # number of frames = number of pages = number of time steps
    datatype = tif.pages[0].asarray().dtype # the compressed tiff file with be save with the same data type

    compressed_tiff_filename = filename + '_compressed_ratio_' + str(R) + file_extension
    with tifffile.TiffWriter(compressed_tiff_filename) as compressed_tiff:
        for frame in range(0,N):
            matrix = tif.pages[frame].asarray().astype('float')
            compressed_matrix = compress_matrix(matrix, compression_ratio)
            compressed_image = compressed_matrix.astype(datatype)
            compressed_tiff.write(compressed_image,contiguous=True)
            
    print('Compressed tiff file saved as ' + compressed_tiff_filename )

In [165]:
data_directory = "/Volumes/DATA/DHM 1082/pani3004/Behnaz_project/Schizophrenic_iPSCs_lines/SZ9322/Test2/Week-2/2022.08.03 16-10-31/Fluo9322_CS2A2S1"
all_tiff_files = []
N = 0
for file in os.listdir(data_directory):
    #if file.startswith("5439CTL_"):
    if file.endswith(".tif"):
        all_tiff_files.append(file)
        N+=1
        #print(os.path.join(data_directory, file))
        
all_names = [] 
for file in all_tiff_files: 
    all_names.append(file[:-4])
#print(all_names)    
    
all_names = sorted(all_names, key = lambda x: int(x[7:]))

all_sorted_tiff_files = [name+'.tif' for name in all_names]

all_sorted_directories = [ data_directory + "/" + file for file in all_sorted_tiff_files ]
all_sorted_directories[0]

'/Volumes/DATA/DHM 1082/pani3004/Behnaz_project/Schizophrenic_iPSCs_lines/SZ9322/Test2/Week-2/2022.08.03 16-10-31/Fluo9322_CS2A2S1/9322SZ_0.tif'

In [166]:
len(all_sorted_directories),N

(841, 841)

In [167]:
tif = tifffile.TiffFile(all_sorted_directories[0])
N = len(tif.pages)
tif.pages[N-1].asarray()

array([[ 8,  6,  9, ...,  7,  7,  5],
       [ 5,  6,  5, ...,  7,  4,  4],
       [ 8,  8,  7, ...,  5,  7,  6],
       ...,
       [16, 10, 10, ...,  7,  5,  4],
       [15, 11,  9, ...,  8,  4,  8],
       [15, 11, 10, ...,  7,  8,  7]], dtype=uint8)

In [168]:
tif.pages[0].shape

(1040, 1392)

In [170]:
prefix = 0
prefix = all_sorted_directories[0][80:-14]
# Replace forward slashes with underscores
prefix = prefix.replace('/', '_')
prefix = prefix.replace(' ', '_')
prefix = prefix.replace('.', '_')
prefix = prefix.replace('-', '_')
print(prefix)


Test2_Week_2_2022_08_03_16_10_31_Fluo9322_CS2A2S


In [171]:
# Specify the output directory
output_directory = '/Volumes/DATA/DHM 1082/yaza3022/hIPSC/Schizopherenic_iPSCs/Fluo/Test2/week2/Fluo_stack/SZ9322/'
output_file = os.path.join(output_directory, prefix + '_stack.tif')

In [172]:

# Define the output file path
with tifffile.TiffWriter(output_file) as big_tif:
    for full_file_name in all_sorted_directories:
        with tifffile.TiffFile(full_file_name) as tif:
            big_tif.write(tif.pages[0].asarray(),contiguous=True)

In [173]:
stack_path = os.path.join(output_directory, prefix + '_stack.tif')
stack_path

'/Volumes/DATA/DHM 1082/yaza3022/hIPSC/Schizopherenic_iPSCs/Fluo/Test2/week2/Fluo_stack/SZ9322/Test2_Week_2_2022_08_03_16_10_31_Fluo9322_CS2A2S_stack.tif'

# Cropping with dark frames

In [174]:
tif = tifffile.imread(stack_path)
N,m,n =  tif.shape
tif.shape

(841, 1040, 1392)

In [175]:
for frame in range(0,N):
    cropped = tif[frame]
    Fluo_cropped = cropped[40:(cropped.shape[0]-120),235:(cropped.shape[1]-277)] #cropped[101:(cropped.shape[0]-189),296:(cropped.shape[1]-346)]
    #Fluo_cropped = Fluo_cropped.astype('uint32')
    imageio.imwrite('/Users/behnaz/NCADD/NCADD/DHM-Fluo images pipeline/test_crop/'+str(frame)+'.tif', Fluo_cropped) 

# Re-stacking cropped fluo (with dark frames)

In [176]:
data_directory = "/Users/behnaz/NCADD/NCADD/DHM-Fluo images pipeline/test_crop"

all_tiff_files = []
N = 0
for file in os.listdir(data_directory):
    if file.startswith(""):
        if file.endswith(".tif"):
            all_tiff_files.append(file)
            N+=1
        #print(os.path.join(data_directory, file))
        
all_names = [] 
for file in all_tiff_files: 
    all_names.append(file[:-4])
    
    
all_names = sorted(all_names, key = lambda x: int(x[:]))
#print(all_names)
all_sorted_tiff_files = [name+'.tif' for name in all_names]
#print(all_sorted_tiff_files)
all_sorted_directories = [ data_directory + "/" + file for file in all_sorted_tiff_files ]
print(len(all_sorted_directories))
# prefix = all_sorted_tiff_files[0][0:11]
output_file = os.path.join(output_directory, prefix + '_cropped_stack.tif')
with tifffile.TiffWriter(output_file) as big_tif:
    for full_file_name in all_sorted_directories:
        with tifffile.TiffFile(full_file_name) as tif:
            big_tif.write(tif.pages[0].asarray(),contiguous=True)

841


In [177]:
output_file

'/Volumes/DATA/DHM 1082/yaza3022/hIPSC/Schizopherenic_iPSCs/Fluo/Test2/week2/Fluo_stack/SZ9322/Test2_Week_2_2022_08_03_16_10_31_Fluo9322_CS2A2S_cropped_stack.tif'

# Compress

In [178]:
compress_tiff(output_file,4)

Compressed tiff file saved as /Volumes/DATA/DHM 1082/yaza3022/hIPSC/Schizopherenic_iPSCs/Fluo/Test2/week2/Fluo_stack/SZ9322/Test2_Week_2_2022_08_03_16_10_31_Fluo9322_CS2A2S_cropped_stack_compressed_ratio_4.tif
